In [10]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os

# Konfigurasi path
DATA_PATH = '../../../stemming/data_preprocessing_final.csv'
INSET_PATH = '../../../../kamus/inset_final.csv'
POLITIK_PATH = '../../../kamus/inset_vader_political_modified.csv'
OUTPUT_DIR = '../../../stemming/outputs/RSN'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [11]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweet")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweet
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertib media online DPR pemerintah jangan spor...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus evaluasi lagi kebijakan bebas visa utama...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang pengaturan logis apa undang un...
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,bebas suara dapat memang jamin UU tetapi bebas...


In [ ]:
# 2.3 Load Leksikon (InSet + Politik) dan Definisi Function Words

# 1. Load InSet Original
df_inset = pd.read_csv(INSET_PATH)
df_inset['kata'] = df_inset['kata'].astype(str).str.strip().str.lower()

# 2. Load Leksikon Politik Baru 
df_politik = pd.read_csv(POLITIK_PATH)
df_politik['kata'] = df_politik['kata'].astype(str).str.strip().str.lower()

# 3. Normalisasi dan Penggabungan (Khusus RSN: InSet * 0.8, Politik Tetap)
df_inset['skor'] = df_inset['skor'] * 0.8
df_politik_ready = df_politik[['kata', 'mean']].rename(columns={'mean': 'skor'})

# Gabungkan: Leksikon Politik akan menimpa InSet jika ada kata yang sama (keep='last')
df_combined_lexicon = pd.concat([df_inset, df_politik_ready]).drop_duplicates(subset='kata', keep='last')

# Ubah ke Dictionary untuk Matching
lexicon_dict = dict(zip(df_combined_lexicon['kata'], df_combined_lexicon['skor']))

# --- DEFINISI KATA FUNGSI ---
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL 
    | KATA_HUBUNG_PREPOSISI 
    | PRONOMINA_DEMONSTRATIVA 
    | PARTIKEL_KATA_TANYA
)

print(f"[INFO] Leksikon berhasil digabungkan. Total vocabulary: {len(lexicon_dict)}")

[INFO] Leksikon berhasil digabungkan. Total vocabulary: 9101


In [13]:
# 2.4 Konfigurasi Remove Set 
REMOVE_SET = KATA_HUBUNG_PREPOSISI | PRONOMINA_DEMONSTRATIVA | PARTIKEL_KATA_TANYA

# Gunakan dataframe gabungan 
df_found = df_combined_lexicon[df_combined_lexicon['kata'].isin(ALL_FUNCTION_WORDS)].copy().sort_values('kata')
total_fw_in_lexicon = len(df_found)

# Ambil kata fungsi yang akan dihapus
remove_set_final = set(df_found[df_found['kata'].isin(REMOVE_SET)]['kata'])

# Menggunakan dictionary gabungan yang sudah skala 4
final_dict = lexicon_dict 

print(f"[DIAGNOSTIK] Total kata fungsi ditemukan di Leksikon Gabungan: {total_fw_in_lexicon}")
print(f"\n[KONFIGURASI] Dictionary Gabungan siap: {len(final_dict)} entri.")
print(f"Kata fungsi yang akan dihapus: {len(remove_set_final)} kata.")

[DIAGNOSTIK] Total kata fungsi ditemukan di Leksikon Gabungan: 21

[KONFIGURASI] Dictionary Gabungan siap: 9101 entri.
Kata fungsi yang akan dihapus: 13 kata.


In [14]:
# 2.5 Menampilkan daftar kata yang akan dihapus
print("\n[DAFTAR] Kata fungsi yang akan dihapus sepenuhnya:")
for word in sorted(remove_set_final):
    print(f"  - {word}")


[DAFTAR] Kata fungsi yang akan dihapus sepenuhnya:
  - aku
  - anda
  - apa
  - dari
  - dia
  - itu
  - karena
  - pada
  - pun
  - saya
  - siapa
  - ya
  - yang


In [15]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)

print(f"\n[INFO] Tokenisasi selesai. Total token awal: {df['tokens'].str.len().sum():,}")


[INFO] Tokenisasi selesai. Total token awal: 235,560


In [16]:
# 2.7 Fungsi Lexicon Matching dengan Penghapusan Kata Fungsi
def match_lexicon_remove_func(tokens, lexicon, remove_set):
    matched_words = []
    unmatched_words = []
    removed_words = []
    
    for token in tokens:
        token_lower = token.lower()
        if token_lower in remove_set:
            removed_words.append(token)
        elif token_lower in lexicon:
            matched_words.append(token)
        else:
            unmatched_words.append(token)
           
    return matched_words, unmatched_words, removed_words

In [17]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching dengan penghapusan kata fungsi...")

# Gunakan final_dict (Gabungan) bukan inset_dict (Original)
df[['matched_words', 'unmatched_words', 'removed_words']] = df['tokens'].apply(
    lambda x: pd.Series(match_lexicon_remove_func(x, final_dict, remove_set_final))
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching dengan penghapusan kata fungsi...
[INFO] Lexicon matching selesai.


In [18]:
# 2.9 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_removed = df['removed_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

# Token sisa setelah penghapusan (konten + negasi/modal)
filtered_words = total_matched + total_unmatched

print("\n[STATISTIK] Hasil Lexicon Matching (Hapus Fungsi):")
print(f"Total token awal         : {total_words:,}")
print(f"Dihapus (kata fungsi)    : {total_removed:,} ({(total_removed/total_words)*100:.2f}%)")
print(f"Token sisa (konten)      : {filtered_words:,}")
print(f"Matched di InSet         : {total_matched:,} ({(total_matched/filtered_words)*100:.2f}% dari token sisa)")
print(f"Unmatched                : {total_unmatched:,} ({(total_unmatched/filtered_words)*100:.2f}% dari token sisa)")
print(f"Coverage Rate (konten)   : {(total_matched/filtered_words)*100:.2f}%")


[STATISTIK] Hasil Lexicon Matching (Hapus Fungsi):
Total token awal         : 235,560
Dihapus (kata fungsi)    : 8,599 (3.65%)
Token sisa (konten)      : 226,961
Matched di InSet         : 114,619 (50.50% dari token sisa)
Unmatched                : 112,342 (49.50% dari token sisa)
Coverage Rate (konten)   : 50.50%


In [19]:
# 2.9.1. Kumpulkan semua kata unmatched dari dataframe hasil matching
from collections import Counter

all_unmatched = []
for lst in df['unmatched_words']:  # Pastikan kolom ini ada
    all_unmatched.extend([w.lower() for w in lst])

# 2. Hitung frekuensi dan ambil top 50
unmatched_freq = Counter(all_unmatched)
top_50_unmatched = unmatched_freq.most_common(50)

# 3. Tampilkan
print("TOP 50 KATA UNMATCHED PALING SERING MUNCUL:")
print(f"{'Kata':<20} | {'Frekuensi':<10}")
print("-" * 35)
for word, freq in top_50_unmatched:
    print(f"{word:<20} | {freq:<10}")

TOP 50 KATA UNMATCHED PALING SERING MUNCUL:
Kata                 | Frekuensi 
-----------------------------------
ri                   | 5492      
di                   | 3158      
dan                  | 3101      
ini                  | 1902      
?                    | 1613      
untuk                | 1515      
dengan               | 1434      
ke                   | 1365      
tahun                | 1127      
ketua                | 1118      
partai               | 1086      
masa                 | 944       
!                    | 932       
jika                 | 765       
akan                 | 704       
ii                   | 703       
oleh                 | 638       
indonesia            | 571       
bagai                | 561       
agenda               | 525       
daerah               | 504       
tetapi               | 486       
i                    | 467       
juga                 | 454       
kepemimpinan         | 440       
kali                 | 436       
sa

In [20]:
# 2.10 Preview Hasil Matching
print("\n[PREVIEW] 5 Tweet Pertama:")
for i in range(5):
    print(f"\nTweet {i+1}: {df['teks_processed'].iloc[i][:80]}...")
    print(f"  Matched : {df['matched_words'].iloc[i]}")
    print(f"  Removed : {df['removed_words'].iloc[i]}")


[PREVIEW] 5 Tweet Pertama:

Tweet 1: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
  Matched : ['ADIL', 'punya', 'kebijakan', 'ingat']
  Removed : ['yang', 'yang']

Tweet 2: tertib media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada...
  Matched : ['tertib', 'DPR', 'pemerintah', 'jangan', 'sporadis', 'selektif', 'hanya']
  Removed : ['yang']

Tweet 3: harus evaluasi lagi kebijakan bebas visa utama untuk negara tiongkok pak ! ! bah...
  Matched : ['harus', 'lagi', 'kebijakan', 'bebas', 'bahaya', 'martabat']
  Removed : []

Tweet 4: jangan ngambang pengaturan logis apa undang undang...
  Matched : ['jangan', 'undang', 'undang']
  Removed : ['apa']

Tweet 5: bebas suara dapat memang jamin UU tetapi bebas sebut tidak harus bablas sehingga...
  Matched : ['bebas', 'suara', 'dapat', 'memang', 'jamin', 'UU', 'bebas', 'tidak', 'harus', 'bablas', 'tabrak']
  Removed : []


In [ ]:
# Create filtered tokens and scores (Removing removed_words physically)
def get_filtered_data(tokens, matched_words, lexicon):
    # filtered_tokens hanya berisi kata yang masuk kategori matched atau unmatched
    # (kata fungsi dalam removed_words benar-benar hilang)
    filtered_tokens = [t for t in tokens if t.lower() in lexicon or t.lower() not in remove_set_final]
    
    # filtered_word_scores_detail untuk kebutuhan rekonstruksi di notebook 03
    filtered_scores = []
    for t in tokens:
        t_low = t.lower()
        if t_low in remove_set_final:
            continue # Lewati kata fungsi
        
        score = lexicon.get(t_low, 0)
        filtered_scores.append({'word': t, 'score': score, 'matched': t_low in lexicon})
            
    return filtered_tokens, filtered_scores

# Terapkan fungsi
df[['filtered_tokens', 'filtered_word_scores']] = df.apply(
    lambda r: pd.Series(get_filtered_data(r['tokens'], r['matched_words'], final_dict)), axis=1
)

# Simpan ke CSV
df.to_csv(OUTPUT_PATH, index=False)

In [21]:
# 2.11 Simpan Output
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, 'lexicon_matching_remove_func.csv')
df.to_csv(output_path, index=False)

print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")
print("[CATATAN] Kolom 'removed_words' berisi kata fungsi yang difilter.Kolom ini digunakan untuk dokumentasi, bukan untuk proses heuristik.")


[OUTPUT] Data berhasil disimpan ke: ../../../stemming/outputs/RSN\lexicon_matching_remove_func.csv
[CATATAN] Kolom 'removed_words' berisi kata fungsi yang difilter.Kolom ini digunakan untuk dokumentasi, bukan untuk proses heuristik.
